# CHAKSU Dataset

This notebook checks the CHAKSU training ZIP, shows one photograph with its expert record, and reports problems. It reads the ZIP without unpacking or changing it.

The labels are NORMAL and GLAUCOMA SUSPECT. A suspect label does not mean confirmed glaucoma. The example uses Expert 1's record; the full checks use the supplied Majority Decision labels.


## 1. Find the photographs and label files

Import the tools used to read the ZIP, read the label tables, and show one photograph.


In [ ]:
import csv
import io
from pathlib import Path
from zipfile import ZipFile
from IPython.display import Image, display

Find the project folder, whether the notebook was opened from there or from `notebooks`.

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

zip_path = project_root / "data/raw/chaksu_train.zip"

Stop with a clear message if the training ZIP is missing.

In [ ]:
if not zip_path.is_file():
    raise FileNotFoundError(
        f"Cannot find {zip_path}. Open from the project or notebooks folder, "
        "and place chaksu_train.zip in data/raw/."
    )

List the files inside the ZIP. Skip folders and extra Mac system files.


In [ ]:
with ZipFile(zip_path) as archive:
    files = [
        item.filename for item in archive.infolist()
        if not item.is_dir() and "__MACOSX" not in item.filename
    ]

Find the original photographs by their folder and file type.


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_files = sorted(
    name for name in files
    if "original_fundus" in name.lower()
    and Path(name).suffix.lower() in image_extensions
)

Print the number of photographs and the names of the first five files.


In [ ]:
print(f"Original photographs found: {len(image_files)}")
print("\nFirst five image paths:")
print("\n".join(image_files[:5]) or "No original photographs found.")

Check that Expert 1's label table is present for each camera. The full checks read all five experts' tables and the majority labels later.


In [ ]:
label_files = [
    f"Train/6.0_Glaucoma_Decision/Expert 1/{camera}.csv"
    for camera in ["Bosch", "Forus", "Remidio"]
]
for name in label_files:
    print(f"{'FOUND' if name in files else 'MISSING'}: {name}")

## 2. Show one photograph and its record

Choose one Bosch photograph and its row in Expert 1's table. Photo numbers can repeat across cameras, so the photograph and label must come from the same camera.


In [ ]:
image_path = "Train/1.0_Original_Fundus_Images/Bosch/Image101.JPG"
label_path = "Train/6.0_Glaucoma_Decision/Expert 1/Bosch.csv"

Read the photograph and its label table from the ZIP.


In [ ]:
with ZipFile(zip_path) as archive:
    image_bytes = archive.read(image_path)
    label_text = archive.read(label_path).decode("utf-8-sig")

The label tables sometimes put more than one filename in a row. Use the same photo-number reader as the full checks to match them correctly.


In [ ]:
import sys

sys.path.insert(0, str(project_root / "data_pipeline"))
from check_chaksu import image_id


Read the label table and find the row with the same photo number.


In [ ]:
records = list(csv.DictReader(io.StringIO(label_text)))
matches = [
    record for record in records
    if image_id(record["Images"]) == image_id(Path(image_path).name)
]


Make sure the photograph has exactly one matching row.


In [ ]:
if len(matches) != 1:
    raise ValueError(f"Expected exactly one matching record, found {len(matches)}.")

Show the photograph and Expert 1's measurements and label. These alone do not confirm a diagnosis.


In [ ]:
display(Image(data=image_bytes, width=450))
print(f"Expert 1's record for {Path(image_path).name}\n")
for field, value in matches[0].items():
    print(f"{field}: {value}")

## 3. Run the full basic checks

Run the same checks on every photograph and label. This also checks the disc and cup masks: images marking those two regions in the eye. It can take a few minutes and reports problems without changing the dataset.


In [ ]:
import runpy

runpy.run_path(str(project_root / "data_pipeline/check_chaksu.py"), run_name="__main__");

## What was found

- The training ZIP has 1,009 photos: 872 normal and 137 glaucoma suspect.
- All photos opened and matched their labels. All 2,018 checked disc and cup masks passed the basic checks.
- There are six pairs of identical photo files.
- We flagged three unusual filenames, a spelling error in the labels, and 41 extra Bosch label rows with no photo in this ZIP. The files were left unchanged.
- We still need to check which photos belong to the same patient before splitting the training data. A suspect label does not mean confirmed glaucoma, so the labels cannot yet be treated as the same as PAPILA's.
